# Driver comparison — bubble charts of the environmental intensities by indicator

1 subplot per indicator, 2 bubbles per item of the X axis: M_dom+M_imp
(French products, size Y_dom) and M_row (imported products, size Y_imp).
See `planefr_lib.plot_bubble` for details.

Two steps:
1. Export of one Excel workbook per scenario (`bubble_data_by_indicator_<scenario>.xlsx`):
   1 sheet per indicator, with a block of rows for the categories of the
   `BRIDGE_SHEET` bridge (on top) and another for the sectors, each sorted
   according to the row chosen in B1 (editable by hand in Excel).
2. Bubble charts read back from these workbooks (`create_driver_comparison`):
   first n consumption categories or first n sectors, baseline alone or compared with
   a transition scenario. Each figure (`bubble_chart_*.png`) is saved with the
   Excel workbook of its data (same name, `.xlsx`).

All the files are written in `figures/driver_comparison/`.

In [ ]:
# ============================ PARAMETERS ============================
SCENARIO_GROUP = "CONF"           # France scenarios (e.g. "CONF", "ADEME", "no_SSP", "SSP2_2050"...)
COMPARISON_SCENARIO = "S1+"       # transition scenario compared with the reference (a SCENARIO_GROUP folder)
BRIDGE_SHEET = "M_split"          # sector grouping of bridge_matrices.xlsx: "M" (5 categories) or "M_split" (14, one per product)
OVERWRITE_EXCEL = True            # True: regenerate bubble_data_*.xlsx; False: keep existing ones (sort row B1 edited by hand)
BUBBLE_TYPES = ["categories", "products"]  # "categories": first N consumption categories; "products": first N sectors
N = 5                             # number of categories / sectors displayed per indicator (None: all)
BUBBLE_COMPARE = [False, True]    # False: reference alone; True: reference vs COMPARISON_SCENARIO
DPI = 300                         # resolution of the saved .png
# ====================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

notebooks_dir = Path.cwd()
while not (notebooks_dir / "planefr_lib").exists() and notebooks_dir != notebooks_dir.parent:
    notebooks_dir = notebooks_dir.parent
if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

import matplotlib.pyplot as plt

from planefr_lib import config, io, processing
from planefr_lib.plot_bubble import export_indicator_tables_excel, load_indicator_tables_excel, create_driver_comparison

# Folder of the figures (and of their Excel workbooks): figures/<NOTEBOOK_NAME>/
NOTEBOOK_NAME = "driver_comparison"
figures_dir = config.figure_dir(NOTEBOOK_NAME)

## 1. Loading the common data

In [ ]:
bridge_m = io.load_bridge_m_matrix(BRIDGE_SHEET)
cf_df = io.load_characterization_factors(SCENARIO_GROUP)
scenario_folders = config.get_scenario_folders(exclude_2019=True, scenario_group=SCENARIO_GROUP)
reference_folder = scenario_folders[config.REFERENCE_SCENARIO_IDX]
comparison_folder = next((f for f in scenario_folders if f.name == COMPARISON_SCENARIO), None)
if comparison_folder is None:
    raise ValueError(f"Scenario '{COMPARISON_SCENARIO}' not found among {[f.name for f in scenario_folders]}")
print(f"Reference scenario: {reference_folder.name} | compared with: {comparison_folder.name}")
print(f"Bridge {BRIDGE_SHEET}: {bridge_m.shape}, categories = {list(bridge_m.columns)}")

## 2. Excel export of the data by indicator

One workbook per scenario, 1 sheet per indicator: a block for the categories
of the `BRIDGE_SHEET` bridge (on top), then a block for the sectors (200
products), with the same rows (x_dom, S_dom, F_x_dom, M_imp, M_dom, M_row, Y_imp,
Y_dom, M*Y products), each sorted in decreasing order of the row chosen in B1.

`OVERWRITE_EXCEL = False`: an existing workbook is not regenerated, to keep
the sort rows (B1) edited by hand in Excel -- unless its categories no longer
match `BRIDGE_SHEET` or it comes from an earlier format. Set to `True` to
recompute everything from the MRIO data.

In [ ]:
def excel_is_up_to_date(path):
    """True if the workbook has the categories of the current bridge and the
    current format (categories sorted by formulas); otherwise it is regenerated."""
    sheets = load_indicator_tables_excel(path)
    expected = [str(c) for c in bridge_m.columns]
    return bool(sheets) and all(s["categories"] == expected and s["categories_sorted"] for s in sheets.values())


excel_paths = {}
for folder in [reference_folder, comparison_folder]:
    path = figures_dir / f"bubble_data_by_indicator_{folder.name}.xlsx"
    if OVERWRITE_EXCEL or not path.exists() or not excel_is_up_to_date(path):
        export_indicator_tables_excel(folder, cf_df, bridge_m, path)
        print(f"Excel table saved: {path}")
    else:
        print(f"Existing Excel table kept: {path}")
    excel_paths[folder.name] = path

base_excel = excel_paths[reference_folder.name]
comparison_excel = excel_paths[comparison_folder.name]

## 3. Bubble charts (read back from the Excel workbooks)

Parameters of `create_driver_comparison`:
- `type`: `"categories"` (consumption categories) or `"products"` (sectors),
  in both cases in decreasing order of the sort row chosen in B1 of each
  Excel sheet;
- `n`: number of categories / sectors displayed (the first `n`; `None`: all);
- `scenarios`: `False` = baseline alone, `True` = comparison with `comparison_excel`
  (columns and order taken from the ranking of the baseline).

In [ ]:
for type_ in BUBBLE_TYPES:
    for compare in BUBBLE_COMPARE:
        suffix = type_ if N is None else f"top{N}_{type_}"
        scope = f"{reference_folder.name}_vs_{comparison_folder.name}" if compare else reference_folder.name
        output_path = figures_dir / f"bubble_chart_{scope}_{suffix}.png"
        fig, _ = create_driver_comparison(base_excel, type=type_, n=N, scenarios=compare,
                                          comparison_excel_path=comparison_excel,
                                          excel_path=output_path.with_suffix(".xlsx"))
        fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
        plt.show()
        print(f"Figure saved: {output_path}")